In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
import anndata

In [ ]:
sc.logging.print_header()
sc.settings.set_figure_params(facecolor="white")

In [ ]:
bdata = sc.read_h5ad(
    "C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/ADP1.h5ad"
)

In [ ]:
top_genes = ['MICU3', 'SLC7A2','RBM14','RBM4','RBM14-RBM4','SLC12A5','MRAS','GALNT8','KCNA6', 'STK24','PRDM4','TRIM23','VPS13C', 'BMPR1B','FNTB','CHURC1-FNTB','RAB15', 'LMO7']

In [ ]:
adata = bdata[:, bdata.var.Feature_ids.isin(top_genes)]

In [ ]:
adata.var

In [ ]:
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata

In [ ]:
adata

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=50)

In [ ]:
adata

In [ ]:
sc.tl.tsne(adata)

In [ ]:
sc.tl.louvain(adata)

In [ ]:
adata

In [ ]:
adata.write('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/BDP1.h5ad')

In [ ]:
adata1 = adata.copy()  # Create a copy of the original object
adata1.var.drop('Feature_ids', axis=1, inplace=True)

In [ ]:
# Filter the data to remove 'Exclude' from the 'disease' column
adata2 = adata1[adata1.obs['disease'] != 'Exclude'].copy()

In [ ]:
adata2.var

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
import anndata

In [ ]:
sc.logging.print_header()
sc.settings.set_figure_params(facecolor="white")

In [ ]:
adata = sc.read_h5ad(
    "C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/BDP1.h5ad"
)

In [ ]:
adata1 = adata.copy()  # Create a copy of the original object
adata1.var.drop('Feature_ids', axis=1, inplace=True)

In [ ]:
# Filter the data to remove 'Exclude' from the 'disease' column
adata2 = adata1[adata1.obs['disease'] != 'Exclude'].copy()

In [ ]:
adata2

In [ ]:
# Define the group order, excluding 'Exclude'
group_order = ["Control", "AsymAD", "AD"]

In [ ]:
# Create dataframe1: cell count for each cell type and disease condition
cell_counts = adata2.obs.groupby(['cell_type', 'disease']).size().unstack(fill_value=0)
# Create dataframe2: sample count for each cell type and disease condition
sample_counts = adata2.obs.groupby(['cell_type', 'disease'])['projid'].nunique().unstack(fill_value=0)
# Save both dataframes to an Excel file
with pd.ExcelWriter('cell_and_sample_counts.xlsx') as writer:
    cell_counts.to_excel(writer, sheet_name='Cell_Counts')
    sample_counts.to_excel(writer, sheet_name='Sample_Counts')

print("DataFrames have been saved to 'cell_and_sample_counts.xlsx'.")

In [ ]:
# Run the rank_genes_groups analysis using the Wilcoxon method
sc.tl.rank_genes_groups(adata2, groupby="cell_type", method="wilcoxon", key_added="wilcoxon_cluster_cell_type")

# Plot the top 16 ranked genes for each group
sc.pl.rank_genes_groups(adata2, n_genes=16, sharey=False, key="wilcoxon_cluster_cell_type")

# Initialize an empty list to hold results for all cell types
all_results = []

# Extract the full results from adata.uns
results = adata2.uns["wilcoxon_cluster_cell_type"]

# Loop through each cell type in adata.obs["cell_type"]
for cell_type in adata2.obs["cell_type"].unique():
    # Loop through each group and extract the results
    for group in results["names"].dtype.names:
        group_names = results["names"][group]
        group_pvals = results["pvals"][group]  # Original p-values (before adjustment)
        group_logfoldchanges = results["logfoldchanges"][group]
        
        # Create a DataFrame for the current group and cell type
        group_data = pd.DataFrame({
            "Gene": group_names,
            "pval": group_pvals,  # Original p-values
            "logfoldchanges": group_logfoldchanges,
            "cell_type": cell_type  # Add cell type to each result row
        })
        
        # Append the current group data to the all_results list
        all_results.append(group_data)

# Concatenate all results into a single DataFrame
Markers_df = pd.concat(all_results, ignore_index=True)

In [ ]:
from statsmodels.stats.multitest import multipletests
# Apply Benjamini-Hochberg correction
Markers_df['pval_adj'] = multipletests(Markers_df['pval'], method='fdr_bh')[1]

In [ ]:
Markers_df.to_excel("C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/data/DATA/cell_type1.xlsx", index=False)

In [ ]:
Markers_df = Markers_df[(Markers_df["pval_adj"] < 0.05) & (abs(Markers_df["logfoldchanges"]) > 0.2630344)]

In [ ]:
Markers_df.to_excel("C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/data/DATA/cell_type2.xlsx", index=False)

In [ ]:
Markers_df

In [ ]:
markers_unique = Markers_df.drop_duplicates(subset="Gene", keep="first")

In [ ]:
markers_unique

In [ ]:
# Define the gene order explicitly
desired_gene_order = ['VPS13C','LMO7','MRAS','MICU3','STK24','CHURC1-FNTB','BMPR1B','RBM4','SLC12A5','PRDM4','GALNT8','TRIM23','RAB15','RBM14-RBM4','SLC7A2','FNTB']

# Ensure the genes in 'top_genes' match the desired order (if not, reorder them)
top_genes = [gene for gene in desired_gene_order if gene in markers_unique["Gene"].values]

# Set the disease categories as an ordered categorical variable
group_order = ["Control", "AsymAD", "AD"]
adata2.obs['disease'] = pd.Categorical(adata2.obs['disease'], categories=group_order, ordered=True)

# Extract the relevant cell types from your markers table
valid_cell_types = Markers_df["cell_type"].unique().tolist()

# Filter the AnnData object to include only the valid cell types
adata2 = adata2[adata2.obs['cell_type'].isin(valid_cell_types)]

# Set the global font size for plots
plt.rcParams['font.size'] = 16

# Create the dotplot with the specified gene order
fig = sc.pl.dotplot(adata2, top_genes, groupby=["cell_type", "disease"], return_fig=True, color_map='Reds')

# Save the figure as a PDF
fig.savefig('dotplot1.pdf', dpi=300)

# Optionally, close the figure after saving
plt.close(fig)


In [ ]:
adata = sc.read_h5ad(
    "C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/ADP2.h5ad"
)

In [ ]:
adata

In [ ]:
plt.rcParams['font.size'] = 42
plt.figure(figsize=(10, 10))
sc.pl.tsne(adata, 
           color=['VPS13C','LMO7','MRAS','MICU3','STK24','CHURC1-FNTB','BMPR1B','RBM4','SLC12A5','PRDM4','GALNT8','TRIM23','RAB15','RBM14-RBM4','SLC7A2','FNTB'],
           cmap='Reds',  # Change the color map to "Reds" (white to red)
           legend_fontsize=42,  # Increase the legend font size
           vmin=0, vmax=4,
           gene_symbols="gene_ids",
           show=False)  # Prevent the figure from showing immediately
plt.savefig('tsne_plot3.pdf', dpi=300, bbox_inches='tight')  # Save as PNG with 300 dpi

In [ ]:
# Filter data to include only "Control" and "AsymAD" samples
adata_filtered = adata2[adata2.obs['disease'].isin(['Control', 'AsymAD'])]

# Define the list of cell types
cell_types = ["Ast", "End", "Ex", "In", "Mic", "Opc", "Oli", "Per"]

# Initialize a list to collect all results
all_results = []

# Step 2: Loop over each cell type and perform differential analysis
for cell_type in cell_types:
    # Subset the data for the specific cell type
    adata_cell_type = adata_filtered[adata_filtered.obs["cell_type"] == cell_type]
    
    # Perform differential expression analysis: AsymAD (case) vs Control 
    sc.tl.rank_genes_groups(
        adata_cell_type, 
        groupby="disease", 
        method="wilcoxon", 
        corr_method="benjamini-hochberg", 
        key_added=f"wilcoxon_{cell_type}",
        reference="Control"  # "Control" is set as the reference (control group)
    )
    
    # Plot the results for the top 16 genes
    sc.pl.rank_genes_groups(
        adata_cell_type, 
        n_genes=16, 
        sharey=False, 
        key=f"wilcoxon_{cell_type}"
    )
    
    # Extract results for each group for the current cell type
    results = adata_cell_type.uns[f"wilcoxon_{cell_type}"]

    # Loop through each group and extract the results
    for group in results["names"].dtype.names:
        group_names = results["names"][group]
        group_scores = results["scores"][group]
        group_pvals = results["pvals"][group]
        group_logfoldchanges = results["logfoldchanges"][group]
        
        # Add the cell type to the group column
        group_data = pd.DataFrame({
            "Gene": group_names,
            "scores": group_scores,
            "pval": group_pvals,
            "logfoldchanges": group_logfoldchanges,
            "group": group,
            "cell_type": cell_type  # Add cell type to each result row
        })
        
        # Append the current cell type's results to the all_results list
        all_results.append(group_data)

# Concatenate all results into a single DataFrame
Markers_df1 = pd.concat(all_results, ignore_index=True)

In [ ]:
from statsmodels.stats.multitest import multipletests
# Apply Benjamini-Hochberg correction
Markers_df1['pval_adj'] = multipletests(Markers_df1['pval'], method='fdr_bh')[1]

In [ ]:
Markers_df1.to_excel("C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/data/DATA/AsymADvsControl.xlsx", index=False)

In [ ]:
adata_filtered = adata2[adata2.obs['disease'].isin(['Control', 'AD'])]

# Define the list of cell types
cell_types = ["Ast", "End", "Ex", "In", "Mic", "Opc", "Oli", "Per"]

# Initialize a list to collect all results
all_results = []

# Step 2: Loop over each cell type and perform differential analysis
for cell_type in cell_types:
    # Subset the data for the specific cell type
    adata_cell_type = adata_filtered[adata_filtered.obs["cell_type"] == cell_type]
    
    # Perform differential expression analysis: AD (case) vs Control
    sc.tl.rank_genes_groups(
        adata_cell_type, 
        groupby="disease", 
        method="wilcoxon", 
        corr_method="benjamini-hochberg", 
        key_added=f"wilcoxon_{cell_type}",
        reference="Control"  # "Control" is set as the reference (control group)
    )
    
    # Plot the results for the top 16 genes
    sc.pl.rank_genes_groups(
        adata_cell_type, 
        n_genes=16, 
        sharey=False, 
        key=f"wilcoxon_{cell_type}"
    )
    
    # Extract results for each group for the current cell type
    results = adata_cell_type.uns[f"wilcoxon_{cell_type}"]

    # Loop through each group and extract the results
    for group in results["names"].dtype.names:
        group_names = results["names"][group]
        group_scores = results["scores"][group]
        group_pvals = results["pvals"][group]
        group_logfoldchanges = results["logfoldchanges"][group]
        
        # Add the cell type to the group column
        group_data = pd.DataFrame({
            "Gene": group_names,
            "scores": group_scores,
            "pval": group_pvals,
            "logfoldchanges": group_logfoldchanges,
            "group": group,
            "cell_type": cell_type  # Add cell type to each result row
        })
        
        # Append the current cell type's results to the all_results list
        all_results.append(group_data)

# Concatenate all results into a single DataFrame
Markers_df2 = pd.concat(all_results, ignore_index=True)

In [ ]:
from statsmodels.stats.multitest import multipletests

# Apply Benjamini-Hochberg correction
Markers_df2['pval_adj'] = multipletests(Markers_df2['pval'], method='fdr_bh')[1]

In [ ]:
Markers_df2.to_excel("C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/data/DATA/ADvsControl.xlsx", index=False)

In [ ]:
# Filter data to include only "AD" and "AsymAD" samples
adata_filtered = adata2[adata2.obs['disease'].isin(['AD', 'AsymAD',])]

# Define the list of cell types
cell_types = ["Ast", "End", "Ex", "In", "Mic", "Opc", "Oli", "Per"]

# Initialize a list to collect all results
all_results = []

# Step 2: Loop over each cell type and perform differential analysis
for cell_type in cell_types:
    # Subset the data for the specific cell type
    adata_cell_type = adata_filtered[adata_filtered.obs["cell_type"] == cell_type]
    
    # Perform differential expression analysis: AD (case) vs AsymAD (control)
    sc.tl.rank_genes_groups(
        adata_cell_type, 
        groupby="disease", 
        method="wilcoxon", 
        corr_method="benjamini-hochberg", 
        key_added=f"wilcoxon_{cell_type}",
        reference="AsymAD"  # "AsymAD" is set as the reference (control group)
    )
    
    # Plot the results for the top 16 genes
    sc.pl.rank_genes_groups(
        adata_cell_type, 
        n_genes=16, 
        sharey=False, 
        key=f"wilcoxon_{cell_type}"
    )
    
    # Extract results for each group  for the current cell type
    results = adata_cell_type.uns[f"wilcoxon_{cell_type}"]

    # Loop through each group and extract the results
    for group in results["names"].dtype.names:
        group_names = results["names"][group]
        group_scores = results["scores"][group]
        group_pvals = results["pvals"][group]
        group_logfoldchanges = results["logfoldchanges"][group]
        
        # Add the cell type to the group column
        group_data = pd.DataFrame({
            "Gene": group_names,
            "scores": group_scores,
            "pval": group_pvals,
            "logfoldchanges": group_logfoldchanges,
            "group": group,
            "cell_type": cell_type  # Add cell type to each result row
        })
        
        # Append the current cell type's results to the all_results list
        all_results.append(group_data)

# Concatenate all results into a single DataFrame
Markers_df3 = pd.concat(all_results, ignore_index=True)

In [ ]:
from statsmodels.stats.multitest import multipletests
# Apply Benjamini-Hochberg correction
Markers_df3['pval_adj'] = multipletests(Markers_df3['pval'], method='fdr_bh')[1]

In [ ]:
Markers_df3.to_excel("C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/data/DATA/ADvsAsymAD.xlsx", index=False)